# Chat com seus documentos (RAG) com Transformers/LLM

Aplicação que responde perguntas **com base em PDFs ou TXTs que você enviar**, usando:

- **Embeddings** (Transformer só encoder) para encontrar os trechos mais relevantes;
- **LLM** (Transformer só decoder) para escrever a resposta a partir desses trechos;
- **Gradio** para a interface web.


**Como funciona (RAG)**
documento → divide em trechos → embeddings → pergunta vira embedding → busca por similaridade → trechos + pergunta vão para o LLM → resposta.

## 1. Instalação

In [ ]:
!pip install -q -U transformers accelerate sentence-transformers pypdf gradio

## 2. Configurações e verificação da GPU


In [ ]:
import os
import torch

USA_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USA_GPU else "cpu"

if USA_GPU:
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    print("O notebook funciona na CPU com um modelo menor, porém bem mais devagar.")

# --- Configurações ---
# configuração de linguagem
MODELO_EMBEDDING = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  
MODELO_LLM = "Qwen/Qwen2.5-3B-Instruct" if USA_GPU else "Qwen/Qwen2.5-0.5B-Instruct"
# caracteres por vetor
TAMANHO_TRECHO = 800        
# caracteres repetidos entre trechos vizinhos
SOBREPOSICAO = 150          
# quantidade de trechos entregues ao LLM
TOP_K = 4                   
# quantidade máxima de tokens da resposta
MAX_TOKENS_RESPOSTA = 400   

GPU detectada: Tesla T4


## 3. Carregar os modelos


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(MODELO_EMBEDDING, device=DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODELO_LLM)
tipo = torch.float16 if USA_GPU else torch.float32
try:
    llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=tipo)
except TypeError:  
    llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, torch_dtype=tipo)
llm.to(DEVICE)
llm.eval()

print("Modelos carregados em:", DEVICE)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (14 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelos carregados em: cuda


## 4. Núcleo do RAG
- `ler_arquivo`: extrai o texto de PDF ou TXT.
- `dividir_em_trechos`: quebra o texto em pedaços com sobreposição.
- `indexar`: transforma cada trecho em vetor (embedding).
- `buscar`: compara a pergunta com os trechos e traz os mais parecidos.
- `perguntar`: monta o prompt com o contexto e pede a resposta ao LLM.

In [ ]:
import numpy as np
from pypdf import PdfReader

BASE = {"trechos": [], "fontes": [], "vetores": None}


def ler_arquivo(caminho):
    if caminho.lower().endswith(".pdf"):
        leitor = PdfReader(caminho)
        return "\n".join((pagina.extract_text() or "") for pagina in leitor.pages)
    with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def dividir_em_trechos(texto, tamanho=TAMANHO_TRECHO, sobreposicao=SOBREPOSICAO):
    texto = " ".join(texto.split())
    trechos, inicio = [], 0
    while inicio < len(texto):
        fim = inicio + tamanho
        trechos.append(texto[inicio:fim])
        if fim >= len(texto):
            break
        inicio = fim - sobreposicao
    return trechos


def indexar(caminhos):
    trechos, fontes = [], []
    for caminho in caminhos:
        nome = os.path.basename(caminho)
        for trecho in dividir_em_trechos(ler_arquivo(caminho)):
            trechos.append(trecho)
            fontes.append(nome)
    if not trechos:
        raise ValueError("Não foi possível extrair texto dos arquivos (PDF escaneado?).")
    vetores = embedder.encode(trechos, batch_size=32, normalize_embeddings=True,
                              show_progress_bar=False)
    BASE["trechos"], BASE["fontes"], BASE["vetores"] = trechos, fontes, vetores
    return len(trechos)


def buscar(pergunta, k=TOP_K):
    q = embedder.encode([pergunta], normalize_embeddings=True)[0]
    notas = BASE["vetores"] @ q            # similaridade do cosseno (vetores já normalizados)
    melhores = np.argsort(-notas)[:k]
    return [(BASE["fontes"][i], BASE["trechos"][i], float(notas[i])) for i in melhores]


@torch.no_grad()
def gerar_resposta(pergunta, contexto):
    mensagens = [
        {"role": "system", "content": (
            "Você é um assistente que responde em português do Brasil usando APENAS o "
            "contexto fornecido. Se a resposta não estiver no contexto, diga que não "
            "encontrou essa informação nos documentos. Seja claro e objetivo.")},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPergunta: {pergunta}"},
    ]
    prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    saida = llm.generate(**entradas, max_new_tokens=MAX_TOKENS_RESPOSTA, do_sample=False)
    novos = saida[0][entradas["input_ids"].shape[1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).strip()


def perguntar(pergunta):
    if BASE["vetores"] is None:
        return "Indexe ao menos um documento antes de perguntar.", ""
    achados = buscar(pergunta)
    contexto = "\n\n".join(f"[{i + 1}] ({fonte}) {trecho}"
                           for i, (fonte, trecho, _) in enumerate(achados))
    resposta = gerar_resposta(pergunta, contexto)
    fontes = "\n\n".join(f"[{i + 1}] {fonte} (similaridade {nota:.2f})\n{trecho}"
                         for i, (fonte, trecho, nota) in enumerate(achados))
    return resposta, fontes

## 5. Teste rápido (sem interface)


In [ ]:
amostra = "amostra.txt"
with open(amostra, "w", encoding="utf-8") as f:
    f.write(
        "Os Transformers foram apresentados em 2017 no artigo Attention Is All You Need, "
        "de Vaswani e colaboradores. A arquitetura usa o mecanismo de autoatenção, que permite "
        "processar todas as palavras de uma frase em paralelo. O modelo BERT usa apenas o encoder "
        "e é bom para classificação, enquanto a família GPT usa apenas o decoder e gera texto."
    )

print("Trechos indexados:", indexar([amostra]))
resposta, fontes = perguntar("Em que ano os Transformers foram apresentados?")
print("\nResposta:", resposta)
print("\nTrechos usados:\n", fontes)

Trechos indexados: 1

Resposta: Os Transformers foram apresentados em 2017.

Trechos usados:
 [1] amostra.txt (similaridade 0.46)
Os Transformers foram apresentados em 2017 no artigo Attention Is All You Need, de Vaswani e colaboradores. A arquitetura usa o mecanismo de autoatenção, que permite processar todas as palavras de uma frase em paralelo. O modelo BERT usa apenas o encoder e é bom para classificação, enquanto a família GPT usa apenas o decoder e gera texto.


## 6. Interface web (Gradio)


In [ ]:
import gradio as gr


def ui_indexar(arquivos):
    if not arquivos:
        return "Envie pelo menos um arquivo PDF ou TXT."
    caminhos = [a if isinstance(a, str) else a.name for a in arquivos]
    try:
        n = indexar(caminhos)
    except Exception as erro:
        return f"Erro ao indexar: {erro}"
    return f"{len(caminhos)} arquivo(s) indexado(s), {n} trechos prontos para consulta."


def ui_perguntar(pergunta):
    if not pergunta or not pergunta.strip():
        return "Digite uma pergunta.", ""
    return perguntar(pergunta)


with gr.Blocks(title="Chat com seus documentos") as demo:
    gr.Markdown("# Chat com seus documentos (RAG)\n"
                "Envie PDFs ou TXTs, clique em **Indexar** e faça perguntas sobre o conteúdo.")
    arquivos = gr.File(label="PDFs ou TXTs", file_count="multiple", file_types=[".pdf", ".txt"])
    btn_indexar = gr.Button("Indexar documentos")
    status = gr.Textbox(label="Status", interactive=False)
    pergunta = gr.Textbox(label="Sua pergunta")
    btn_perguntar = gr.Button("Perguntar", variant="primary")
    resposta = gr.Textbox(label="Resposta", lines=8)
    fontes = gr.Textbox(label="Trechos usados na resposta", lines=10)

    btn_indexar.click(ui_indexar, inputs=arquivos, outputs=status)
    btn_perguntar.click(ui_perguntar, inputs=pergunta, outputs=[resposta, fontes])
    pergunta.submit(ui_perguntar, inputs=pergunta, outputs=[resposta, fontes])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8981026534e95c0ad.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
